In [1]:
# ==========================================================
# 1. LIBRARIES
# ==========================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.feature_selection import (
mutual_info_classif,
RFE
)

from sklearn.linear_model import LogisticRegression

from imblearn.over_sampling import SMOTE

# ==========================================================
# 2. LOAD FEATURE-ENGINEERED DATASET
# ==========================================================

CustomerData_Df = pd.read_csv(
"customer_churn_feature_engineered.csv"
)

print("=" * 60)
print("FEATURE-ENGINEERED DATASET")
print("=" * 60)

print("\nDataset Shape:")
print(CustomerData_Df.shape)

print("\nFirst 5 Rows:")
print(CustomerData_Df.head())

# ==========================================================
# 3. INITIAL VALIDATION
# ==========================================================

print("\n" + "=" * 60)
print("INITIAL DATA VALIDATION")
print("=" * 60)

print("\nMissing Values:")
print(CustomerData_Df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(CustomerData_Df.duplicated().sum())

print("\nTarget Distribution:")
print(CustomerData_Df["churn"].value_counts())

print("\nTarget Distribution (%):")
print(
(
CustomerData_Df["churn"]
.value_counts(normalize=True)
* 100
).round(2)
)

# ==========================================================
# 4. SEPARATE FEATURES AND TARGET
# ==========================================================

X = CustomerData_Df.drop(
"churn",
axis=1
)

y = CustomerData_Df["churn"]

print("\n" + "=" * 60)
print("FEATURES AND TARGET")
print("=" * 60)

print("\nFeature Shape:")
print(X.shape)

print("\nTarget Shape:")
print(y.shape)

# ==========================================================
# 5. TRAIN / TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

X,

y,

test_size=0.20,

random_state=42,

stratify=y

)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("\nTraining Shape:")
print(X_train.shape)

print("\nTesting Shape:")
print(X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

# ==========================================================
# 6. IDENTIFY COLUMN TYPES
# ==========================================================

categorical_cols = X_train.select_dtypes(
include=["object", "string"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
include=np.number
).columns.tolist()

print("\n" + "=" * 60)
print("COLUMN TYPES")
print("=" * 60)

print("\nCategorical Variables:")
print(categorical_cols)

print("\nNumber of Categorical Variables:")
print(len(categorical_cols))

print("\nNumerical Variables:")
print(numerical_cols)

print("\nNumber of Numerical Variables:")
print(len(numerical_cols))

# ==========================================================
# 7. NUMERICAL IMPUTATION
# ==========================================================

num_imputer = SimpleImputer(
strategy="median"
)

X_train_num = pd.DataFrame(

num_imputer.fit_transform(
    X_train[numerical_cols]
),

columns=numerical_cols,

index=X_train.index

)

X_test_num = pd.DataFrame(

num_imputer.transform(
    X_test[numerical_cols]
),

columns=numerical_cols,

index=X_test.index

)

# ==========================================================
# 8. CATEGORICAL IMPUTATION
# ==========================================================

cat_imputer = SimpleImputer(
strategy="most_frequent"
)

X_train_cat = pd.DataFrame(

cat_imputer.fit_transform(
    X_train[categorical_cols]
),

columns=categorical_cols,

index=X_train.index

)

X_test_cat = pd.DataFrame(

cat_imputer.transform(
    X_test[categorical_cols]
),

columns=categorical_cols,

index=X_test.index

)

# ==========================================================
# 9. ONE-HOT ENCODING
# ==========================================================

X_train_cat = pd.get_dummies(
X_train_cat,
drop_first=True
)

X_test_cat = pd.get_dummies(
X_test_cat,
drop_first=True
)

# ==========================================================
# 10. ALIGN TRAINING AND TEST COLUMNS
# ==========================================================

X_train_cat, X_test_cat = X_train_cat.align(

X_test_cat,

join="left",

axis=1,

fill_value=0

)

# ==========================================================
# 11. COMBINE NUMERICAL AND CATEGORICAL FEATURES
# ==========================================================

X_train_processed = pd.concat(

[
    X_train_num,
    X_train_cat
],

axis=1

)

X_test_processed = pd.concat(

[
    X_test_num,
    X_test_cat
],

axis=1

)


X_test_processed = X_test_processed[
X_train_processed.columns
]

print("\n" + "=" * 60)
print("PROCESSED DATA")
print("=" * 60)

print("\nProcessed Training Shape:")
print(X_train_processed.shape)

print("\nProcessed Testing Shape:")
print(X_test_processed.shape)

print(
"\nTraining and Testing Columns Match:"
)

print(
list(X_train_processed.columns)
==
list(X_test_processed.columns)
)

# ==========================================================
# 12. FEATURE SCALING
# ==========================================================

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(

scaler.fit_transform(
    X_train_processed
),

columns=X_train_processed.columns,

index=X_train_processed.index

)

X_test_scaled = pd.DataFrame(

scaler.transform(
    X_test_processed
),

columns=X_test_processed.columns,

index=X_test_processed.index

)

print("\n" + "=" * 60)
print("FEATURE SCALING")
print("=" * 60)

print("\nScaled Training Shape:")
print(X_train_scaled.shape)

print("\nScaled Testing Shape:")
print(X_test_scaled.shape)

# ==========================================================
# 13. SAVE PREPROCESSING OBJECTS
# ==========================================================

joblib.dump(
num_imputer,
"numerical_imputer.pkl"
)

joblib.dump(
cat_imputer,
"categorical_imputer.pkl"
)

joblib.dump(
scaler,
"standard_scaler.pkl"
)

print("\nPreprocessing objects saved successfully.")

# ==========================================================
# 14. CLASS DISTRIBUTION BEFORE SMOTE
# ==========================================================

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION BEFORE SMOTE")
print("=" * 60)

print(
y_train.value_counts()
)

# ==========================================================
# 15. APPLY SMOTE
# ==========================================================

sm = SMOTE(
random_state=42
)

X_train_smote, y_train_smote = sm.fit_resample(

X_train_scaled,

y_train

)

print("\n" + "=" * 60)
print("CLASS DISTRIBUTION AFTER SMOTE")
print("=" * 60)

print(
pd.Series(
y_train_smote
).value_counts()
)

print("\nTraining Shape After SMOTE:")
print(
X_train_smote.shape
)

# ==========================================================
# 16. RANDOM FOREST FEATURE IMPORTANCE
# ==========================================================

rf_selector = RandomForestClassifier(

n_estimators=200,

random_state=42,

n_jobs=-1

)

rf_selector.fit(

X_train_smote,

y_train_smote

)

rf_importance = pd.DataFrame({

"Feature":
X_train_smote.columns,

"Importance":
rf_selector.feature_importances_

})

rf_importance = rf_importance.sort_values(

by="Importance",

ascending=False

)

print("\n" + "=" * 60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

print(
rf_importance.head(20)
)

# ==========================================================
# 17. MUTUAL INFORMATION
# ==========================================================

mi_score = mutual_info_classif(

X_train_smote,

y_train_smote,

random_state=42

)

mi_results = pd.DataFrame({

"Feature":
X_train_smote.columns,

"MI_Score":
mi_score

})

mi_results = mi_results.sort_values(

by="MI_Score",

ascending=False

)

print("\n" + "=" * 60)
print("MUTUAL INFORMATION FEATURE RANKING")
print("=" * 60)

print(
mi_results.head(20)
)

# ==========================================================
# 18. RECURSIVE FEATURE ELIMINATION (RFE)
# ==========================================================

log_model = LogisticRegression(

max_iter=1000,

random_state=42

)

rfe = RFE(

estimator=log_model,

n_features_to_select=20

)

rfe.fit(

X_train_smote,

y_train_smote

)

# ==========================================================
# 19. EXTRACT RFE SELECTED FEATURES
# ==========================================================

rfe_features = X_train_smote.columns[
    rfe.support_
].tolist()

print("\n" + "=" * 60)
print("RFE SELECTED FEATURES")
print("=" * 60)

for i, feature in enumerate(
    rfe_features,
    start=1
):
    print(
        f"{i:02d}. {feature}"
    )

print(
    "\nNumber of RFE Selected Features:",
    len(rfe_features)
)

# ==========================================================
# 20. RFE FEATURE RANKING
# ==========================================================

rfe_results = pd.DataFrame({

"Feature":
X_train_smote.columns,

"RFE_Ranking":
rfe.ranking_,

"Selected":
rfe.support_

})

rfe_results = rfe_results.sort_values(

by=[
    "Selected",
    "RFE_Ranking"
],

ascending=[
    False,
    True
]

)

print("\n" + "=" * 60)
print("RFE FEATURE RANKING")
print("=" * 60)

print(
rfe_results.to_string(index=False)
)

# ==========================================================
# 21. CREATE RFE-SELECTED TRAINING DATA
# ==========================================================

X_train_rfe = X_train_smote[
rfe_features
].copy()

# ==========================================================
# 22. CREATE RFE-SELECTED TEST DATA
# ==========================================================

X_test_rfe = X_test_scaled[
rfe_features
].copy()

print("\n" + "=" * 60)
print("RFE DATASET")
print("=" * 60)

print("\nRFE Training Shape:")
print(
X_train_rfe.shape
)

print("\nRFE Testing Shape:")
print(
X_test_rfe.shape
)

# ==========================================================
# 23. VALIDATE RFE FEATURES
# ==========================================================

print("\nRFE Training Columns Match Test Columns:")

print(
list(X_train_rfe.columns)
==
list(X_test_rfe.columns)
)

# ==========================================================
# 24. SAVE FULL-FEATURE DATASET
# ==========================================================

X_train_smote.to_csv(
"X_train_full_smote.csv",
index=False
)

X_test_scaled.to_csv(
"X_test_full.csv",
index=False
)

pd.Series(
y_train_smote,
name="churn"
).to_csv(
"y_train_smote.csv",
index=False
)

y_test.reset_index(
drop=True
).to_csv(
"y_test.csv",
index=False
)

# ==========================================================
# 25. SAVE RFE-SELECTED DATASET
# ==========================================================

X_train_rfe.to_csv(
"X_train_rfe.csv",
index=False
)

X_test_rfe.to_csv(
"X_test_rfe.csv",
index=False
)

# ==========================================================
# 26. SAVE RFE FEATURE LIST
# ==========================================================

pd.DataFrame({

"Feature":
rfe_features

}).to_csv(

"RFE_Selected_Features.csv",

index=False

)

# ==========================================================
# 27. SAVE FEATURE SELECTION RESULTS
# ==========================================================

rf_importance.to_csv(

"RandomForest_Feature_Importance.csv",

index=False

)

mi_results.to_csv(

"MutualInformation_Feature_Ranking.csv",

index=False

)

rfe_results.to_csv(

"RFE_Feature_Ranking.csv",

index=False

)

# ==========================================================
# 28. FINAL VALIDATION
# ==========================================================

print("\n" + "=" * 60)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 60)

print("\nFull Training Dataset:")
print(
X_train_smote.shape
)

print("\nFull Test Dataset:")
print(
X_test_scaled.shape
)

print("\nRFE Training Dataset:")
print(
X_train_rfe.shape
)

print("\nRFE Test Dataset:")
print(
X_test_rfe.shape
)

print("\nTraining Class Distribution After SMOTE:")
print(
pd.Series(
y_train_smote
).value_counts()
)

print("\nOriginal Test Class Distribution:")
print(
y_test.value_counts()
)

print("\nMissing Values - Full Training:")
print(
X_train_smote.isnull().sum().sum()
)

print("\nMissing Values - Full Test:")
print(
X_test_scaled.isnull().sum().sum()
)

print("\nMissing Values - RFE Training:")
print(
X_train_rfe.isnull().sum().sum()
)

print("\nMissing Values - RFE Test:")
print(
X_test_rfe.isnull().sum().sum()
)

# ==========================================================
# 29. COMPLETION MESSAGE
# ==========================================================

print("\n" + "=" * 60)
print("FEATURE SELECTION AND PREPROCESSING COMPLETED")
print("=" * 60)

print("\nFiles created:")

print("\nFull Feature Dataset:")
print(" X_train_full_smote.csv")
print(" X_test_full.csv")

print("\nRFE Feature Dataset:")
print(" X_train_rfe.csv")
print(" X_test_rfe.csv")

print("\nTarget Files:")
print(" y_train_smote.csv")
print(" y_test.csv")

print("\nFeature Selection Results:")
print(" RandomForest_Feature_Importance.csv")
print(" MutualInformation_Feature_Ranking.csv")
print(" RFE_Feature_Ranking.csv")
print(" RFE_Selected_Features.csv")

print("\nPreprocessing Objects:")
print(" numerical_imputer.pkl")
print(" categorical_imputer.pkl")
print(" standard_scaler.pkl")

FEATURE-ENGINEERED DATASET

Dataset Shape:
(10000, 45)

First 5 Rows:
   gender  age     country      city customer_segment  tenure_months  \
0    male   68  bangladesh    london              sme             22   
1  female   57      canada    sydney       individual              9   
2    male   24     germany  new york              sme             58   
3    male   49   australia     dhaka       individual             19   
4    male   65  bangladesh     delhi       individual             52   

  signup_channel contract_type  monthly_logins  weekly_active_days  ...  \
0            web       monthly              26                   7  ...   
1         mobile       monthly               7                   5  ...   
2            web        yearly              19                   5  ...   
3         mobile        yearly              34                   7  ...   
4            web       monthly              20                   6  ...   

   revenue_per_login  complaint_severity  acti

In [6]:
import sys

!{sys.executable} -m pip install --upgrade imbalanced-learn

  Attempting uninstall: imbalanced-learn━━━━━━━━ 0/2 [sklearn-compat]
    Found existing installation: imbalanced-learn 0.13.0━━━━━━━━━━ 1/2 [imbalanced-learn]
    Uninstalling imbalanced-learn-0.13.0:m━━━━━━━━━━━━━━━━━━━ 1/2 [imbalanced-learn]
      Successfully uninstalled imbalanced-learn-0.13.0━━━━━━━━ 1/2 [imbalanced-learn]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn][imbalanced-learn]
